In [1]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

from ultralytics import YOLO

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

PROJECT_ROOT = Path.cwd().parent

PROCESSED_DATA = (
    PROJECT_ROOT
    / "datasets"
    / "processed"
)

EXPERIMENTS_DIR = (
    PROJECT_ROOT
    / "experiments"
)

MODELS_DIR = (
    PROJECT_ROOT
    / "models"
)

EVALUATION_DIR = (
    EXPERIMENTS_DIR
    / "evaluation"
)

EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DEVICE = (
    0
    if torch.cuda.is_available()
    else "cpu"
)

print("Inspectra — Model Evaluation")
print("=" * 60)
print("Device:", DEVICE)

Inspectra — Model Evaluation
Device: 0


In [2]:
model_paths = {
    "bottle": (
        EXPERIMENTS_DIR
        / "bottle"
        / "baseline"
        / "baseline"
        / "weights"
        / "best.pt"
    ),

    "pcb": (
        EXPERIMENTS_DIR
        / "pcb"
        / "baseline"
        / "baseline"
        / "weights"
        / "best.pt"
    ),

    "road": (
        MODELS_DIR
        / "road"
        / "baseline_resnet18.pt"
    ),
}

for dataset, path in model_paths.items():

    print(
        f"{dataset:10} "
        f"{'FOUND' if path.exists() else 'MISSING'}"
    )

bottle     FOUND
pcb        FOUND
road       FOUND


In [3]:
def evaluate_yolo_model(
    dataset_name,
    model_path
):

    dataset_root = (
        PROCESSED_DATA
        / dataset_name
    )

    yaml_path = (
        dataset_root
        / "data.yaml"
    )

    test_images = (
        dataset_root
        / "test"
        / "images"
    )

    if not model_path.exists():
        print(
            f"{dataset_name}: model not found"
        )
        return None

    if not yaml_path.exists():
        print(
            f"{dataset_name}: data.yaml not found"
        )
        return None

    if not test_images.exists():
        print(
            f"{dataset_name}: test set not found"
        )
        return None

    model = YOLO(
        str(model_path)
    )

    start = time.perf_counter()

    metrics = model.val(
        data=str(yaml_path),
        split="test",
        imgsz=640,
        batch=4,
        device=DEVICE,
        plots=True,
        project=str(
            EVALUATION_DIR
        ),
        name=f"{dataset_name}_baseline",
        exist_ok=True,
        verbose=True,
    )

    elapsed = (
        time.perf_counter()
        - start
    )

    result = {
        "dataset": dataset_name,
        "task": "detection",
        "precision": float(
            metrics.box.mp
        ),
        "recall": float(
            metrics.box.mr
        ),
        "map50": float(
            metrics.box.map50
        ),
        "map50_95": float(
            metrics.box.map
        ),
        "evaluation_seconds": elapsed,
    }

    return result

In [4]:
bottle_metrics = evaluate_yolo_model(
    "bottle",
    model_paths["bottle"]
)

bottle_metrics

Ultralytics 8.4.117  Python-3.11.5 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
WARNING val: Slow image access detected (ping: 0.40.1 ms, read: 6.01.2 MB/s, size: 41.7 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning D:\Inspectra\datasets\processed\bottle\test\labels... 920 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 920/920 449.9it/s 2.0s0.1s
val: New cache created: D:\Inspectra\datasets\processed\bottle\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 230/230 18.3it/s 12.5s0.1s
                   all        920       1829      0.967      0.971      0.982      0.896
                   Cap        468        473       0.97      0.967      0.987      0.879
               Missing        348  

{'dataset': 'bottle',
 'task': 'detection',
 'precision': 0.9670109594891115,
 'recall': 0.9709702968514925,
 'map50': 0.9820735036274845,
 'map50_95': 0.8963879891309094,
 'evaluation_seconds': 34.56459829999949}

In [5]:
pcb_metrics = evaluate_yolo_model(
    "pcb",
    model_paths["pcb"]
)

pcb_metrics

Ultralytics 8.4.117  Python-3.11.5 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
Model summary (fused): 73 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
WARNING val: Slow image access detected (ping: 0.50.1 ms, read: 17.03.3 MB/s, size: 125.0 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning D:\Inspectra\datasets\processed\pcb\test\labels... 829 images, 239 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1068/1068 383.3it/s 2.8s0.1s
val: New cache created: D:\Inspectra\datasets\processed\pcb\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 267/267 19.7it/s 13.6s0.1s
                   all       1068       1662       0.98      0.984      0.988      0.555
            mouse_bite        131        262      0.977      0.985      0.988      0.541
                  spur        138  

{'dataset': 'pcb',
 'task': 'detection',
 'precision': 0.9795733243367813,
 'recall': 0.9839463915129786,
 'map50': 0.9879771306637831,
 'map50_95': 0.5548077556115211,
 'evaluation_seconds': 33.02965869999025}

In [6]:
detection_results = [
    result
    for result in [
        bottle_metrics,
        pcb_metrics,
    ]
    if result is not None
]

detection_df = pd.DataFrame(
    detection_results
)

display(
    detection_df
)

,dataset,task,precision,recall,map50,map50_95,evaluation_seconds
0,bottle,detection,0.967011,0.970970,0.982074,0.896388,34.564598
1,pcb,detection,0.979573,0.983946,0.987977,0.554808,33.029659


In [7]:
from torchvision.models import resnet18
from torchvision.models import ResNet18_Weights
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [9]:
road_model = resnet18(
    weights=None
)

road_model.fc = torch.nn.Linear(
    road_model.fc.in_features,
    2
)

road_device = (
    f"cuda:{DEVICE}"
    if isinstance(DEVICE, int)
    else DEVICE
)

road_state = torch.load(
    model_paths["road"],
    map_location=road_device
)

road_model.load_state_dict(
    road_state
)

road_model = road_model.to(
    road_device
)

road_model.eval()

print(
    "Road model loaded"
)

print(
    "Device:",
    road_device
)

Road model loaded
Device: cuda:0


C:\Users\Garvit\AppData\Local\Temp\ipykernel_25732\679446352.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  road_state = torch.load(


In [10]:
road_transform = transforms.Compose([
    transforms.Resize(
        (224, 224)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    ),
])

road_test = datasets.ImageFolder(
    PROCESSED_DATA
    / "road"
    / "test",
    transform=road_transform
)

road_test_loader = DataLoader(
    road_test,
    batch_size=64,
    shuffle=False,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
)

print(
    "Classes:",
    road_test.classes
)

print(
    "Test images:",
    len(road_test)
)

Classes: ['Negative', 'Positive']
Test images: 6000


In [11]:
all_predictions = []
all_targets = []

with torch.no_grad():

    for images, labels in road_test_loader:

        images = images.to(
            DEVICE
        )

        outputs = road_model(
            images
        )

        predictions = (
            outputs.argmax(
                dim=1
            )
            .cpu()
            .numpy()
        )

        all_predictions.extend(
            predictions
        )

        all_targets.extend(
            labels.numpy()
        )

all_predictions = np.array(
    all_predictions
)

all_targets = np.array(
    all_targets
)

print(
    "Predictions:",
    len(all_predictions)
)

print(
    "Targets:",
    len(all_targets)
)

Predictions: 6000
Targets: 6000


In [12]:
road_accuracy = accuracy_score(
    all_targets,
    all_predictions
)

road_precision = precision_score(
    all_targets,
    all_predictions,
    average="binary"
)

road_recall = recall_score(
    all_targets,
    all_predictions,
    average="binary"
)

road_f1 = f1_score(
    all_targets,
    all_predictions,
    average="binary"
)

road_metrics = {
    "dataset": "road",
    "task": "classification",
    "accuracy": road_accuracy,
    "precision": road_precision,
    "recall": road_recall,
    "f1": road_f1,
}

road_metrics

{'dataset': 'road',
 'task': 'classification',
 'accuracy': 0.999,
 'precision': 1.0,
 'recall': 0.998,
 'f1': 0.998998998998999}

In [13]:
print(
    classification_report(
        all_targets,
        all_predictions,
        target_names=road_test.classes,
        digits=4
    )
)

              precision    recall  f1-score   support

    Negative     0.9980    1.0000    0.9990      3000
    Positive     1.0000    0.9980    0.9990      3000

    accuracy                         0.9990      6000
   macro avg     0.9990    0.9990    0.9990      6000
weighted avg     0.9990    0.9990    0.9990      6000



In [14]:
cm = confusion_matrix(
    all_targets,
    all_predictions
)

plt.figure(
    figsize=(7, 6)
)

plt.imshow(
    cm
)

plt.xticks(
    range(len(road_test.classes)),
    road_test.classes
)

plt.yticks(
    range(len(road_test.classes)),
    road_test.classes
)

plt.xlabel(
    "Predicted"
)

plt.ylabel(
    "Actual"
)

plt.title(
    "Road Classification Confusion Matrix"
)

for i in range(cm.shape[0]):

    for j in range(cm.shape[1]):

        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

plt.colorbar()

plt.show()

<Figure size 700x600 with 2 Axes>

In [15]:
all_results = []

all_results.extend(
    detection_results
)

all_results.append(
    road_metrics
)

results_df = pd.DataFrame(
    all_results
)

display(
    results_df
)

results_df.to_csv(
    EVALUATION_DIR
    / "baseline_results.csv",
    index=False
)

with open(
    EVALUATION_DIR
    / "baseline_results.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        all_results,
        file,
        indent=4
    )

,dataset,task,precision,recall,map50,map50_95,evaluation_seconds,accuracy,f1
0,bottle,detection,0.967011,0.970970,0.982074,0.896388,34.564598,NaN,NaN
1,pcb,detection,0.979573,0.983946,0.987977,0.554808,33.029659,NaN,NaN
2,road,classification,1.000000,0.998000,NaN,NaN,NaN,0.999,0.998999


In [16]:
if not detection_df.empty:

    plt.figure(
        figsize=(10, 5)
    )

    plt.bar(
        detection_df["dataset"],
        detection_df["map50_95"]
    )

    plt.ylabel(
        "mAP50-95"
    )

    plt.xlabel(
        "Dataset"
    )

    plt.title(
        "Detection Baseline Comparison"
    )

    plt.show()

<Figure size 1000x500 with 1 Axes>